### Semantic search with Pinecone

In [ ]:
import pinecone, openai

# Initialize Pinecone
pinecone.init(api_key="<your-pinecone-api-key>", environment="<your-pinecone-environment>")

# Step 1: Create Pinecone index
index_name = "semantic-search-demo"
pinecone.create_index(index_name, dimension=1536, metric="cosine")
index = pinecone.Index(index_name)

# Step 2: Embed some documents (using OpenAI for simplicity)
openai.api_key = "<your-openai-api-key>"
documents = [
    "Pinecone is a vector database used in RAG systems.",
    "Retrieval-augmented generation improves LLM accuracy.",
    "Semantic search uses embeddings to find similar text."
]

def get_embedding(text):
    response = openai.Embedding.create(input=text, engine="text-embedding-ada-002")
    return response['data'][0]['embedding']

# Step 3: Upsert documents into Pinecone
doc_ids = [f"doc-{i}" for i in range(len(documents))]
vectors = [(doc_ids[i], get_embedding(documents[i])) for i in range(len(documents))]
index.upsert(vectors)

# Step 4: Semantic search with query
query = "What database is used in retrieval augmented generation?"
query_vector = get_embedding(query)

# Retrieve top 2 most similar documents
results = index.query(vector=query_vector, top_k=2, include_metadata=False)

# Step 5: Display results
print("Top results:")
for match in results['matches']:
    print(f"- {match['id']} (score: {match['score']:.4f})")